# Retomada das execuções (seeds 44 a 46)

Antes de rodar: suba `runner.py`, `diagnostico.py` e `datasets_meta.py` (pasta `TCC_AutoML_Experimentos/common/`) para `/content/` pelo painel de arquivos do Colab.

Rode as células em ordem. **Não mova nada para a pasta `refazer`** — isso já foi feito uma vez e refazê-lo apaga o progresso. O runner pula sozinho o que já existe no Drive.

In [ ]:
# 1) Mesmas versões das seeds 42-43 (fixadas para não misturar versões entre repetições)
!pip install -q --default-timeout=200 autogluon.tabular==1.5.0 catboost==1.2.10 flaml==2.7.0 h2o==3.46.0.12 tpot==1.1.0
# Se o Colab pedir, reinicie a sessão (Ambiente de execução > Reiniciar sessão) e rode daqui para baixo.

In [ ]:
# 2) Confere se os 3 arquivos estão em /content
import sys, os, glob, collections
sys.path.append("/content")
faltam = [f for f in ["runner.py", "diagnostico.py", "datasets_meta.py"] if not os.path.exists(f"/content/{f}")]
print("Arquivos faltando em /content:", faltam or "nenhum")

In [ ]:
# 3) Drive + registro do ambiente desta sessão (versões, Python, CPU/GPU) em arquivo separado
from google.colab import drive
drive.mount("/content/drive")
DIR = "/content/drive/MyDrive/TCC_resultados"
os.makedirs(DIR, exist_ok=True)
from diagnostico import registrar
registrar(f"{DIR}/ambiente_retomada.json")
# ATENÇÃO: se gpu_disponivel for False, esta sessão não tem T4 (o H2O usou GPU nas seeds 42-43).

In [ ]:
# 4) Progresso atual no Drive (cada seed completa = 32 execuções)
c = collections.Counter(f.split("seed")[1].split(".")[0] for f in glob.glob(f"{DIR}/*__seed*.json"))
for s in range(42, 47):
    print(f"seed {s}: {c.get(str(s), 0)}/32")

In [ ]:
# 5) Retoma o lote (seed por seed; pula o que já existe)
import runner
print("TPOT_PARAMS:", runner.TPOT_PARAMS)  # deve ser {'cv': 3, 'max_eval_time_mins': 1}
TOOLS = ["flaml", "autogluon", "h2o", "tpot"]
DATASETS = ["abalone", "concrete", "wine_quality", "california",
            "bike_sharing", "brazilian_houses", "diamonds", "mercedes"]
runner.executar_tudo(TOOLS, DATASETS, seeds=range(42, 47), budget=300, dir_=DIR)